# Генерация синтетических данных с использованием Faker

In [1]:
!pip install faker

In [2]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

In [3]:
# Инициализация генератора (русская локаль для реалистичных ФИО, адресов и т.д.)
fake = Faker('ru_RU')
Faker.seed(42)      # фиксируем seed для воспроизводимости
random.seed(42)

## Генерация синтетических данных для варианта "Сотрудники компании"

In [4]:
# DataFrame с отделами
def generate_departments(n):
   
    departments = []
    for i in range(1, n + 1):
        departments.append({
            'department_id': i,
            'name': fake.unique.job() + ' отдел',  # уникальное название
            'location': fake.city()
        })
    return pd.DataFrame(departments)

# DataFrame с должностями и диапазоном зарплат
def generate_positions(n: int) -> pd.DataFrame:
    
    positions = []
    for i in range(1, n + 1):
        min_salary = random.randint(30000, 60000)
        max_salary = min_salary + random.randint(20000, 80000)
        positions.append({
            'position_id': i,
            'name': fake.unique.job(),
            'min_salary': min_salary,
            'max_salary': max_salary
        })
    return pd.DataFrame(positions)

# генерируем сотрудников со случайными атрибутами
def generate_employees(n: int, department_ids: list, position_ids: list) -> pd.DataFrame:
    
    employees = []
    for emp_id in range(1, n + 1):        
        first_name = fake.first_name()
        last_name = fake.last_name()
        patronymic = fake.middle_name()
        birth_date = fake.date_of_birth(minimum_age=18, maximum_age=70)
        phone = fake.phone_number()
        email = fake.email()
        address = fake.address().replace('\n', ', ')
        
        # дата найма: не раньше 18-летия и не позже сегодняшнего дня
        min_hire_date = birth_date + timedelta(days=18*365)
        hire_date = fake.date_between(start_date=min_hire_date, end_date='today')
        
        # статус -  active / terminated (10% уволены)
        status = random.choices(['active', 'terminated'], weights=[0.9, 0.1])[0]
        
        # выбор отдела и должности
        department_id = random.choice(department_ids)
        position_id = random.choice(position_ids)
        
        # зарплата     
        salary = random.randint(40000, 150000)
        
        employees.append({
            'employee_id': emp_id,
            'last_name': last_name,
            'first_name': first_name,
            'patronymic': patronymic,
            'birth_date': birth_date,
            'gender': random.choice(['М', 'Ж']),
            'address': address,
            'phone': phone,
            'email': email,
            'hire_date': hire_date,
            'status': status,
            'department_id': department_id,
            'position_id': position_id,
            'salary': salary
        })
    return pd.DataFrame(employees)

# создать историю зарплат на основе данных сотрудников
def generate_salary_history(employees_df: pd.DataFrame, avg_records: int = 3):    
    history = []
    record_id = 1
    
    for _, emp in employees_df.iterrows():
        emp_id = emp['employee_id']
        hire_date = emp['hire_date']
        current_salary = emp['salary']
        
        # количество изменений (0 – если сотрудник только нанят и ещё не было изменений)
        num_changes = random.choices([0, 1, 2, 3, 4], weights=[0.2, 0.3, 0.3, 0.1, 0.1])[0]
        
        # генерируем даты изменений (после hire_date и до сегодня)
        change_dates = sorted([fake.date_between(start_date=hire_date, end_date='today') 
                               for _ in range(num_changes)])
        
        # начальная зарплата при найме (можно сделать немного отличающейся от текущей)
        #  будем считать, что первая запись – это зарплата при найме,
        # а последующие – повышения.
        if num_changes == 0:
            # Если изменений не было, всё равно добавим одну запись (начальная)
            history.append({
                'history_id': record_id,
                'employee_id': emp_id,
                'change_date': hire_date,
                'new_salary': current_salary
            })
            record_id += 1
        else:
            # генерируем возрастающие зарплаты
            salary_values = sorted([random.randint(30000, current_salary) for _ in range(num_changes)])
            # добавляем текущую зарплату как последнюю
            salary_values.append(current_salary)
            # даты: hire_date и change_dates
            all_dates = [hire_date] + change_dates
            for i in range(len(all_dates)):
                history.append({
                    'history_id': record_id,
                    'employee_id': emp_id,
                    'change_date': all_dates[i],
                    'new_salary': salary_values[i]
                })
                record_id += 1
    return pd.DataFrame(history)

In [5]:
N_DEPARTMENTS = 10
N_POSITIONS = 20
N_EMPLOYEES = 500

In [6]:
departments_df = generate_departments(N_DEPARTMENTS)
positions_df = generate_positions(N_POSITIONS)
employees_df = generate_employees(
    N_EMPLOYEES,
    departments_df['department_id'].tolist(),
    positions_df['position_id'].tolist()
)

In [7]:
departments_df

,department_id,name,location
0,1,Диетолог отдел,клх Кырен
1,2,Метеоролог отдел,п. Волоколамск
2,3,Гепатолог отдел,г. Артем
3,4,Гомеопат отдел,п. Териберка
4,5,Бактериолог отдел,клх Усинск
5,6,Танцор отдел,с. Чикола
6,7,Нотариус отдел,ст. Ессентуки
7,8,Тележурналист отдел,к. Ельня
8,9,Литейщик отдел,г. Видное
9,10,Сиделка отдел,к. Нефедова


In [8]:
positions_df

,position_id,name,min_salary,max_salary
0,1,Мясник,50952,78248
1,2,Борт-радист,30819,99417
2,3,Философ,39012,75061
3,4,Доярка,37314,66458
4,5,Семейный врач,54132,80849
5,6,Врач скорой помощи,52174,120714
6,7,Ортопед,59234,114975
7,8,Реставратор,32848,91546
8,9,Композитор,43825,65907
9,10,Военный прокурор,30976,57116


In [9]:
employees_df

,employee_id,last_name,first_name,patronymic,birth_date,gender,address,phone,email,hire_date,status,department_id,position_id,salary
0,1,Гаврилов,Мина,Кузьминична,1973-07-29,М,"к. Бологое, пер. Островского, д. 77, 242388",+7 928 327 64 83,tatjana_03@example.org,2009-06-10,active,7,11,76421
1,2,Григорьева,Ольга,Федотович,1995-02-12,Ж,"д. Красногорск (Моск.), ш. Заречное, д. 684 ст...",8 532 871 01 22,erjabova@example.com,2023-09-28,active,6,4,52156
2,3,Крюкова,Лаврентий,Егоровна,1989-12-15,М,"п. Хасавюрт, ул. Вахитова, д. 382, 824896",83252880957,makarovamaja@example.com,2015-02-24,active,6,20,74671
3,4,Лукина,Тимур,Николаевна,2002-05-09,М,"г. Яхрома, алл. Профсоюзная, д. 951, 473829",+7 (465) 787-1331,adrian_98@example.org,2025-03-29,active,9,4,89615
4,5,Мартынов,Савватий,Станиславовна,1987-08-05,М,"п. Уварово, алл. Челюскинцев, д. 84 к. 81, 080132",8 311 656 67 01,maslovalukija@example.org,2019-05-29,active,10,12,115674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,Кузнецова,Твердислав,Теймуразович,1992-04-15,Ж,"с. Кажим, бул. Ручейный, д. 1 стр. 1, 669579",+7 (996) 872-25-89,semendementev@example.org,2010-12-05,active,4,16,112173
496,497,Фомина,Амвросий,Валерьевич,1966-11-01,М,"клх Котельнич, алл. Угольная, д. 4/4 стр. 356,...",+78255442867,ikarpov@example.com,1996-12-04,active,8,16,100401
497,498,Гуляев,Мефодий,Владленович,1994-07-26,Ж,"к. Ардон, пр. Механический, д. 9/6 стр. 895, 4...",+7 (693) 358-95-13,vatslav_2008@example.net,2014-08-05,active,6,6,58344
498,499,Корнилова,Евдокия,Якубович,1994-09-02,М,"п. Партизанск, ул. 40 лет Победы, д. 8 к. 455,...",+7 179 720 38 59,karpevseev@example.org,2020-10-01,active,9,2,108674


In [10]:
# Генерация истории зарплат
salary_history_df = generate_salary_history(employees_df, avg_records=3)
salary_history_df

,history_id,employee_id,change_date,new_salary
0,1,1,2009-06-10,71333
1,2,1,2024-01-04,76421
2,3,2,2023-09-28,34948
3,4,2,2025-03-22,52156
4,5,3,2015-02-24,74671
...,...,...,...,...
1291,1292,498,2022-10-03,57426
1292,1293,498,2024-04-22,58344
1293,1294,499,2020-10-01,89827
1294,1295,499,2022-11-11,108674


In [11]:
# Сохранение в CSV
departments_df.to_csv('departments.csv', index=False)
positions_df.to_csv('positions.csv', index=False)
employees_df.to_csv('employees.csv', index=False)
salary_history_df.to_csv('salary_history.csv', index=False)

# Задание на самостоятельную работу
Создать синтетический набор данных, сохранить его в csv файлы, построить на основе данных онтологию, вывести онтограф.


### Вариант 1. Аренда электромобилей
Сущности: 
- Автомобиль: id, марка, модель, год выпуска, госномер, запас хода (км), статус (доступен, арендован, на обслуживании), тариф (цена за минуту), локация (парковка). История зарядок (JSON-массив объектов: {id станции, дата_начала, дата_конца, объём_кВт·ч, стоимость}).
- Клиент: id, ФИО, водительское удостоверение (серия, номер), телефон, email, дата регистрации.
- Аренда:id, id клиента, id автомобиля, дата и время начала, дата и время окончания, начальный пробег, конечный пробег, стоимость, статус (активна, завершена, отменена), Информация об оплате (поля внутри таблицы): сумма, способ оплаты, дата оплаты, статус оплаты.
- Зарядная станция: id, адрес, тип разъёма, количество мест, статус (работает/не работает).

Объём данных: автомобили - 100, клиенты - 500, аренды - 3000, зарядные станции -  20

Онтология:
- Классы: Автомобиль, Клиент, Аренда, ЗаряднаяСтанция.
- Связи:  клиент арендует автомобиль, автомобиль участвует в арендах, история зарядок хранится в JSON внутри автомобиля, ссылаясь на станции, Аренда содержит данные об оплате.

### Вариант 2. Банковские клиенты и транзакции
Сущности: 
- Клиент: id, ФИО, дата рождения, паспортные данные (серия, номер, кем выдан), адрес, телефон, email.
- Счёт: id, номер счёта, тип (дебетовый/кредитный), валюта, дата открытия, остаток, id клиента.
- Транзакция: id, дата, сумма, тип (пополнение/списание), id счёта, описание.
- Кредит: id, id клиента, сумма, процентная ставка, срок, дата выдачи, ежемесячный платёж.

Объём: 1000 клиентов, у каждого 1–3 счёта, на каждом 10–50 транзакций, 20% клиентов имеют кредиты.

Онтология:
- Классы: Клиент, Счёт, Транзакция, Кредит.
- Связи: клиент владеет счетами, по счетам проходят транзакции, клиент может иметь кредиты.

### Вариант 3. Медицинские пациенты и приёмы
Сущности:
- Пациент: id, ФИО, дата рождения, пол, полис ОМС, СНИЛС, адрес, телефон.
- Врач: id, ФИО, специальность, кабинет.
- Приём: id, дата и время, id пациента, id врача, диагноз (код МКБ-10), жалобы, назначения (JSON-массив объектов: {лекарство, дозировка, курс}).
- Больничный лист: id, id приёма, дата начала, дата окончания, диагноз.

Объём: 2000 пациентов, 50 врачей, 5–10 приёмов на пациента за последние 2 года, часть с больничными.

Онтология:
- Классы: Пациент, Врач, Приём, БольничныйЛист.
- Связи: пациент посещает врача (приём), по приёму может быть открыт больничный, назначения хранятся в JSON приёма (Атрибут класса Прием).

### Вариант 4. Студенты и успеваемость
Сущности:
- Студент: id, ФИО, дата рождения, группа, год поступления, форма обучения.
- Группа: id, номер, курс, институт, направление подготовки.
- Предмет: id, название, семестр, количество часов, форма контроля (экзамен/зачёт), id преподавателя.
- Оценка: id, id студента, id предмета, дата, оценка (число или зачёт/незачёт).
- Преподаватель: id, ФИО, кафедра.

Объём: 10 групп по 25 студентов = 250 студентов, 30 предметов, у каждого студента оценки по всем предметам (≈ 7500 записей).

Онтология
- Классы: Студент, Группа, Предмет, Оценка, Преподаватель.
- Связи: студент учится в группе, группа изучает предметы (связка группа-предмет), преподаватель ведёт предмет, студент получает оценки.

### Вариант 5. Интернет-магазин: товары, заказы, покупатели
Сущности:
- Покупатель: id, email, ФИО, телефон, дата регистрации.
- Товар: id, название, категория, цена, остаток на складе.
- Заказ: id, id покупателя, дата, статус (новый, оплачен, отгружен, доставлен).
- Состав заказа: id заказа, id товара, количество, цена в момент заказа.
- Отзыв: id, id покупателя, id товара, оценка (1–5), текст, дата.

Объём: 500 покупателей, 1000 товаров, 3000 заказов (в среднем по 2 товара в заказе), 1500 отзывов.

Онтология:
- Классы: Покупатель, Товар, Заказ, СтрокаЗаказа, Отзыв.
- Связи: покупатель делает заказ, заказ содержит товары (с количеством), покупатель оставляет отзыв на товар.

### Вариант 6. Недвижимость и сделки
Сущности: 
- Объект недвижимости: id, адрес, тип (квартира/дом/коммерческое), площадь, количество комнат, этаж, год постройки, кадастровый номер.
- Владелец: id, ФИО, паспортные данные, доля (если долевая собственность).
- Сделка купли-продажи: id, дата, id объекта, id продавца, id покупателя, цена, нотариус.
- Ценовой архив: id объекта, дата изменения, новая цена (для отслеживания динамики).

Объём: 2000 объектов, 300 владельцев, 150 сделок, у каждого объекта 2–3 записи цен.

Онтология:
- Классы: Объект, Владелец, Сделка, ЦеновойАрхив.
- Связи: объект может иметь нескольких владельцев (доли), сделка связывает продавца, покупателя и объект.

### Вариант 7. Транспортные средства и штрафы
Сущности:
- Владелец ТС: id, ФИО, водительское удостоверение (серия, номер), адрес, телефон.
- Автомобиль: id, госномер, марка, модель, год выпуска, VIN, цвет, id владельца.
- Штраф: id, id автомобиля, дата, статья нарушения, сумма, статус (оплачен/не оплачен).
- Страховой полис: id, id автомобиля, компания, дата начала, дата окончания, стоимость.

Объём: 1000 владельцев, 1200 автомобилей (некоторые владеют несколькими), 5000 штрафов, 1500 полисов.

Онтология:
- Классы: Владелец, Автомобиль, Штраф, Полис.
- Связи: владелец владеет автомобилями, автомобиль имеет штрафы и страховки.

### Вариант 8. Библиотека: книги, читатели, выдачи
Сущности:
- Книга: id, ISBN, название, автор, год издания, издательство, жанр, количество экземпляров в библиотеке.
- Читатель: id, ФИО, дата рождения, адрес, телефон, email, дата регистрации.
- Выдача: id, id книги, id читателя, дата выдачи, дата возврата (если есть), фактическая дата возврата.
- Бронь: id, id книги, id читателя, дата брони, статус (активна/исполнена).

Объём: 500 книг, 300 читателей, 2000 выдач, 100 броней.

Онтология:
- Классы: Книга, Читатель, Выдача, Бронь.
- Связи: читатель берёт книгу (выдача), может бронировать.

### Вариант 9. Техподдержка: заявки и обращения
Сущности:
- Пользователь: id, ФИО, email, телефон, уровень (обычный/премиум).
- Категория заявки: id, название, описание.
- Сотрудник поддержки: id, ФИО, отдел, количество решённых заявок.
- Заявка: id, id пользователя, id категории, тема, описание, дата создания, статус (новая, в работе, решена, закрыта), приоритет, id сотрудника (назначен), комментарии (JSON-массив объектов: {id автора (пользователь или сотрудник), текст, дата}).

Объём данных: пользователи - 1000, категории - 5, сотрудники - 10, заявки - 5000, комментарии - 15 000  (общее на все заявки)

Онтология:
- Классы: Пользователь, Категория, Сотрудник, Заявка.
- Связи: пользователь создаёт заявку; заявка относится к категории; заявка назначается сотруднику; комментарии принадлежат заявке (в JSON).

### Вариант 10. Доставка обедов из ресторанов
Сущности: 
- Ресторан: id, название, адрес, кухня (тип кухни), рейтинг.
- Блюдо: id, название, id ресторана, категория, цена, вес/порция, калории.
- Клиент: id, ФИО, телефон, email, адрес доставки (основной), дата регистрации.
- Заказ: id, id клиента, id ресторана, дата и время заказа, статус (принят, готовится, доставлен, отменён), общая сумма, способ оплаты, состав заказа (JSON-массив объектов: {id блюда, количество, цена_в_момент_заказа}), информация о доставке (поля внутри той же таблицы): курьер (ФИО), дата и время назначения курьера, дата и время фактической доставки, статус доставки (назначен, в пути, выполнен), фактический адрес доставки (если отличается от адреса клиента)

Объём данных: рестораны -  20, блюда - в среднем 20 на ресторан, клиенты -  1000, заказы - 5000.

Онтология: 
- Классы: Ресторан, Блюдо, Клиент, Заказ.
- Связи: ресторан предлагает блюда, клиент оформляет заказ, заказ ссылается на ресторан и клиента.

Состав заказа и данные доставки хранятся внутри сущности «Заказ» (денормализовано с использованием JSON).


# Выполнение задания Вариант 7 Транспортные средства и штрафы

Сущности:
- Владелец ТС: id, ФИО, водительское удостоверение (серия, номер), адрес, телефон.
- Автомобиль: id, госномер, марка, модель, год выпуска, VIN, цвет, id владельца.
- Штраф: id, id автомобиля, дата, статья нарушения, сумма, статус (оплачен/не оплачен).
- Страховой полис: id, id автомобиля, компания, дата начала, дата окончания, стоимость.

Объём: 1000 владельцев, 1200 автомобилей (некоторые владеют несколькими), 5000 штрафов, 1500 полисов.

Онтология:
- Классы: Владелец, Автомобиль, Штраф, Полис.
- Связи: владелец владеет автомобилями, автомобиль имеет штрафы и страховки.

## Генерация синтетических данных

In [2]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

In [3]:
fake = Faker('ru_RU')
Faker.seed(42)      # фиксируем seed для воспроизводимости
random.seed(42)

### Владельцы авто

id, ФИО, водительское удостоверение (серия, номер), адрес, телефон.

In [4]:
N_CAR_OWNERS = 1000

In [5]:
# DataFrame с владельцами автомобилей
def generate_car_owners(n):
   
    car_owners = []
    for i in range(1, n + 1):

        used_licenses = set()

        while True:
            dl = f"{fake.random_int(10, 99)} {fake.random_int(10, 99)} {fake.random_int(100000, 999999)}"
            if dl not in used_licenses:
                used_licenses.add(dl)
                break

        car_owners.append({
            'owner_id': i,
            'name': fake.name(),
            'drive_license': dl,
            'address': fake.address().replace('\n', ', '),
            'phone': fake.phone_number()
        })
    return pd.DataFrame(car_owners)

<div>
<img src="image.png" width="500"/>
</div>

In [6]:
car_owners_df = generate_car_owners(N_CAR_OWNERS)
car_owners_df

,owner_id,name,drive_license,address,phone
0,1,Николаева Елена Валентиновна,91 24 126225,"д. Ребриха, ул. Студенческая, д. 93, 838637",84026542351
1,2,Мечислав Евсеевич Константинов,21 58 201414,"д. Гремячинск (Бурят.), ул. Кирова, д. 4 к. 50...",+7 (553) 419-28-32
2,3,Тамара Дмитриевна Зимина,69 58 383060,"п. Архыз, пр. Флотский, д. 639 к. 77, 242388",+79696532871
3,4,Вишнякова Наталья Эльдаровна,16 24 260265,"с. Шамары, пер. Ударный, д. 518 к. 2, 627048",+7 (814) 893-25-28
4,5,Адам Брониславович Гущин,77 10 728038,"п. Беслан, пер. Крестьянский, д. 971, 278248",86383465787
...,...,...,...,...,...
995,996,Титов Тихон Фёдорович,87 38 325294,"г. Кизилюрт, наб. Куйбышева, д. 602 стр. 4/9, ...",8 (181) 580-61-94
996,997,Пономарева Милица Максимовна,87 82 908164,"к. Горно-Алтайск, ул. Минская, д. 9/8 стр. 886...",+7 (932) 850-71-96
997,998,Авдеев Вышеслав Артемьевич,20 29 719699,"д. Новая Игирма, алл. Западная, д. 1, 371775",8 (856) 274-8066
998,999,Евгения Николаевна Устинова,23 18 824047,"клх Кириши, пр. Степана Разина, д. 865, 211860",+7 811 239 93 30


### Автомобили

id, госномер, марка, модель, год выпуска, VIN, цвет, id владельца.

In [7]:
N_CARS = 1200

In [8]:
from faker_vehicle import VehicleProvider
fake.add_provider(VehicleProvider)

In [9]:
# DataFrame с автомобилями
def generate_cars(n):
   
    cars = []
    for i in range(1, n + 1):
        year, make, model = (fake.vehicle_year_make_model().split(' ', 2)) #получение полной информации об авто

        owner_ids = list(range(1, N_CAR_OWNERS + 1)) #первые 1000 машин принадлежат первым 1000 владельцам
        extra = [random.randint(1, N_CAR_OWNERS) for _ in range(N_CARS - N_CAR_OWNERS)] #рандомные владельцы для оставшихся 200 машин
        owner_ids.extend(extra)
        #random.shuffle(owner_ids) # перемешиваем, чтобы не было очевидной связи между id машины и владельца

        cars.append({
            'car_id': i,
            'car_plate': fake.license_plate(),
            'year': year,
            'brand': make,
            'model': model,
            'vin': fake.vin(),
            'color': fake.color_name(),
            'owner_id': owner_ids[i - 1] # каждый владелец должен иметь хотя бы 1 машину
        })
    return pd.DataFrame(cars)

In [10]:
cars_df = generate_cars(N_CARS)
cars_df

,car_id,car_plate,year,brand,model,vin,color,owner_id
0,1,ОA641 03,2016,MINI,Paceman,GGT35Y7GXULGH0834,Светло-зеленый,1
1,2,AО9580 72,2009,Nissan,Maxima,9WGMSUEN80H6K9074,Коралловый,2
2,3,000T495 177,2018,Mercedes-Benz,Sprinter 3500 XD Cargo,VUBLCJWS9NC376919,Серый,3
3,4,РE1746 85,2011,Chevrolet,Express 3500 Cargo,L19R9VR60SU1C1662,Темно-бирюзовый,4
4,5,4689УУ 72,2011,Suzuki,Grand Vitara,X38SPRAYX5PY97402,Темно-голубой,5
...,...,...,...,...,...,...,...,...
1195,1196,B5117 55,2000,Ford,Excursion,TDJH4T400RBUT1091,Каштановый,317
1196,1197,Р7833 32,2007,Mercedes-Benz,C-Class,BRDYC5TB4Z5934369,Лазурный,193
1197,1198,Р2671 64,2006,Lexus,GX,U6KW15AD26RHR9454,Аквамарин,267
1198,1199,У5006 67,1999,Volkswagen,Cabrio,NAMC43GLXMB005786,Шоколадный,365


In [11]:
cars_df.owner_id.value_counts()

owner_id
998    4
267    3
199    3
123    3
162    3
      ..
996    1
997    1
125    1
999    1
875    1
Name: count, Length: 1000, dtype: int64

### Штраф

id, id автомобиля, дата, статья нарушения, сумма, статус (оплачен/не оплачен).

In [12]:
N_FINES = 5000

In [13]:
# DataFrame с автомобилями
def generate_fines(n):
    violation_articles = [
    "Статья 12.9 ч.2 (Превышение скорости на 20-40 км/ч)",
    "Статья 12.12 ч.1 (Проезд на красный свет)",
    "Статья 12.19 ч.3 (Остановка на пешеходном переходе)",
    "Статья 12.16 ч.4 (Выезд на встречную полосу)",
    "Статья 12.6 (Непристегнутый ремень безопасности)",
    "Статья 12.8 ч.1 (Управление ТС в состоянии опьянения)",
    "Статья 12.18 (Непредоставление преимущества пешеходу)",
    "Статья 12.36.1 (Использование телефона за рулем)"
]
    fines = []
    for i in range(1, n + 1):

        fines.append({
            'fine_id': i,
            'car_id': random.randint(1, N_CARS),
            'date': fake.date_this_year(),
            'article': random.choice(violation_articles),
            'amount': random.randrange(500, 5001, 500),
            'status': random.choice(['Оплачен', 'Не оплачен'])
        })
    return pd.DataFrame(fines)

In [14]:
fines_df = generate_fines(N_FINES)
fines_df

,fine_id,car_id,date,article,amount,status
0,1,946,2026-04-17,Статья 12.19 ч.3 (Остановка на пешеходном пере...,4000,Не оплачен
1,2,785,2026-04-01,Статья 12.19 ч.3 (Остановка на пешеходном пере...,2500,Оплачен
2,3,377,2026-02-03,Статья 12.8 ч.1 (Управление ТС в состоянии опь...,2000,Оплачен
3,4,567,2026-01-14,Статья 12.18 (Непредоставление преимущества пе...,3500,Оплачен
4,5,297,2026-04-08,Статья 12.12 ч.1 (Проезд на красный свет),1500,Не оплачен
...,...,...,...,...,...,...
4995,4996,982,2026-04-27,Статья 12.9 ч.2 (Превышение скорости на 20-40 ...,3000,Не оплачен
4996,4997,805,2026-05-03,Статья 12.12 ч.1 (Проезд на красный свет),5000,Не оплачен
4997,4998,54,2026-02-05,Статья 12.9 ч.2 (Превышение скорости на 20-40 ...,4500,Оплачен
4998,4999,223,2026-01-04,Статья 12.8 ч.1 (Управление ТС в состоянии опь...,2500,Оплачен


### Страховой полис

id, id автомобиля, компания, дата начала, дата окончания, стоимость.

In [15]:
from collections import Counter

In [16]:
N_POLICIES = 1500

In [17]:
def generate_insurance_policies(n_policies):
    policies = []


    car_ids = list(range(1, N_CARS + 1))
                                                                # из 1500:
    insured_cars = random.sample(car_ids, k=int(N_CARS * 0.9)) # 1080 машин будут застрахованы, остальные 120 - нет
    extra_policies = random.choices(insured_cars, k= n_policies - len(insured_cars)) # 1500 - 1080 =  420 полисов будут случайно распределены между застрахованными машинами

    companies = ['Росгосстрах', 'Ингосстрах', 'АльфаСтрахование', 'РЕСО-Гарантия', 'ВСК', 'Т-Страхование']

    policy_counts = Counter(insured_cars) # считаем, сколько полисов приходится на каждую застрахованную машину (вид: {car_id: count_policies})
    for car in extra_policies:
        policy_counts[car] += 1

    policy_id = 1

    for car_id, count in policy_counts.items():
    # Чтобы полисы не пересекались, определяем базовую дату для этой машины.
    # Если полисов 3, то первый начался примерно 3-4 года назад.
        base_start_date = fake.date_time_between(start_date=f'-{count+1}y', end_date=f'-{count}y') #радномно сгенерировать дату в промежутке от (count+1) лет назад до count лет назад (в расчёте на 1 год действия полиса)
        for i in range(count):
            # Каждый следующий полис начинается через год после предыдущего
            start_date = base_start_date + timedelta(days=365 * i)
            end_date = start_date + timedelta(days=365)
            
            policies.append({
                'policy_id': policy_id,
                'car_id': car_id,
                'company': random.choice(companies),
                'start_date': start_date,
                'end_date': end_date,
                'cost': random.randint(5000, 25000)
            })
            policy_id += 1
    return pd.DataFrame(policies)

In [18]:
policies_df = generate_insurance_policies(N_POLICIES)
policies_df

,policy_id,car_id,company,start_date,end_date,cost
0,1,826,АльфаСтрахование,2024-02-10 20:33:03,2025-02-09 20:33:03,13642
1,2,826,РЕСО-Гарантия,2025-02-09 20:33:03,2026-02-09 20:33:03,18198
2,3,657,Т-Страхование,2024-05-16 09:19:12,2025-05-16 09:19:12,16558
3,4,657,РЕСО-Гарантия,2025-05-16 09:19:12,2026-05-16 09:19:12,11853
4,5,790,Ингосстрах,2023-03-09 19:23:55,2024-03-08 19:23:55,18605
...,...,...,...,...,...,...
1495,1496,107,Росгосстрах,2024-05-05 08:21:42,2025-05-05 08:21:42,24629
1496,1497,107,Т-Страхование,2025-05-05 08:21:42,2026-05-05 08:21:42,22956
1497,1498,908,АльфаСтрахование,2025-05-06 14:22:44,2026-05-06 14:22:44,16844
1498,1499,795,ВСК,2024-08-16 22:09:21,2025-08-16 22:09:21,13393


In [19]:
car_owners_df.to_csv('car_owners.csv', index=False)
cars_df.to_csv('cars.csv', index=False)
fines_df.to_csv('fines.csv', index=False)
policies_df.to_csv('policies.csv', index=False)

## Создание онтологии

In [20]:
from owlready2 import *

In [21]:
onto = get_ontology("http://test.org/transport_v1.owl")

### Классы, связи, атрибуты

In [22]:
with onto:
    #классы
    class Владелец(Thing): pass
    class Автомобиль(Thing): pass
    class Штраф(Thing): pass
    class Полис(Thing): pass

    #связи
    class владеет_автомобилем(ObjectProperty):
        domain = [Владелец]     # Кто владеет
        range = [Автомобиль]    # Чем владеет

    class имеет_штраф(ObjectProperty):
        domain = [Автомобиль]   # У кого штраф (штраф выписывается на машину)
        range = [Штраф]

    class имеет_полис(ObjectProperty):
        domain = [Автомобиль]
        range = [Полис]
        
    #атрибуты
    class фио_владельца(DataProperty):
        domain = [Владелец]
        range = [str]
        
    class госномер(DataProperty):
        domain = [Автомобиль]
        range = [str]

    class сумма_штрафа(DataProperty):
        domain = [Штраф]
        range = [int]

    class статус_штрафа(DataProperty):
        domain = [Штраф]
        range = [str]  # "оплачен" или "не оплачен"

### Индивиды

In [23]:
owners_df = pd.read_csv('car_owners.csv')
cars_df = pd.read_csv('cars.csv') 
fines_df = pd.read_csv('fines.csv')
policies_df = pd.read_csv('policies.csv')

In [24]:
def populate_ontology(owners_df, cars_df, fines_df, policies_df):
    with onto:
        owner_instances = {}
        car_instances = {}

        # владельцы
        for _, row in owners_df.iterrows():
            owner = Владелец(f"владелец_{row['owner_id']}") #владелец_15
            owner.фио_владельца = [row['name']]
            owner_instances[row['owner_id']] = owner

        # автомобили + связь с владельцем
        for _, row in cars_df.iterrows():
            car = Автомобиль(f"авто_{row['car_id']}")
            car.госномер = [row['car_plate']]
            car_instances[row['car_id']] = car

            # владелец -> владеет -> Авто
            if row['owner_id'] in owner_instances:
                owner = owner_instances[row['owner_id']]
                owner.владеет_автомобилем.append(car)

        # штрафы + связь с автомобилем
        for _, row in fines_df.iterrows():
            fine = Штраф(f"штраф_{row['fine_id']}")
            fine.сумма_штрафа = [int(row['amount'])]
            fine.статус_штрафа = [row['status']]
            
            # авто -> имеет_штраф -> Штраф
            if row['car_id'] in car_instances:
                car = car_instances[row['car_id']]
                car.имеет_штраф.append(fine)

        # полисы
        for _, row in policies_df.iterrows():
            policy = Полис(f"полис_{row['policy_id']}")      
            # авто -> имеет_полис -> Полис
            if row['car_id'] in car_instances:
                car = car_instances[row['car_id']]
                car.имеет_полис.append(policy)

In [25]:
populate_ontology(owners_df, cars_df, fines_df, policies_df)

In [26]:
test_owner = onto.search_one(iri="*владелец_10")
if test_owner:
    print(f"\n{test_owner.фио_владельца[0]} владеет автомобилями:")
    for car in test_owner.владеет_автомобилем:
        print(f" - {car.госномер[0]}")


Антонова Эмилия Станиславовна владеет автомобилями:
 - Р5263 152
